# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [39]:
# Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {
    "table": ["employees", "salary", "studies", "employee"],  # ENTER A TABLE COLUMNS HERE,
    "definition": ["Employee information, name...", "Salary details for each year", "Educational studies, name of the institution, type of studies, level", "Single employee"],  # ENTER A TABLE DEFINITATIONS HERE
}
df = pd.DataFrame(data)
print(df)

       table                                         definition
0  employees                      Employee information, name...
1     salary                       Salary details for each year
2    studies  Educational studies, name of the institution, ...
3   employee                                    Single employee


In [42]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])
print(text_tables)

employees: Employee information, name...
salary: Salary details for each year
studies: Educational studies, name of the institution, type of studies, level
employee: Single employee


In [43]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [44]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="Give me the employees information and salaries")#ENTER YOUR QUERY HERE
print(return_OAI(pqt1))

{
    "tables": ["employees", "salary"]
}


In [49]:
pqt3 = """
Given the following tables and their content definitions,
###Tables
{tables}

Return all the tables with their definitions in the following JSON format:
{{
    "tables": {{
        "employees": "Employee information",
        "salary": "Salary details for each year",
        "studies": "Educational studies",
        "employee": "Single employee"
    }}
}}

###User Question:
Give me all the tables with their definitions
""".format(tables=text_tables)

In [51]:
question3 = prompt_question_tables.format(tables=pqt3, question="Show me the single employee") #ENTER YOUR QUERY HERE
print(return_OAI(question3))

{
    "tables": {
        "employee": "Single employee"
    }
}


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## Report: Findings on Table Selection with GPT-3.5-Turbo

### Overview

In this notebook, we explored the ability of GPT-3.5-Turbo to select relevant SQL tables based solely on table names and brief definitions, given a user query. The workflow involved constructing prompts that listed available tables and their descriptions, then asking the model to identify which tables were necessary to answer specific questions.

### Findings

#### Prompt Variations and Model Performance

Several prompt variations were tested, ranging from straightforward requests ("Give me the employees information and salaries") to more complex or ambiguous queries. In most cases, GPT-3.5-Turbo was able to correctly identify the relevant tables based on the provided definitions. For example, when asked for employee information and salaries, the model selected both the `employees` and `salary` tables, which was the expected behavior.

However, some variations revealed limitations:

- **Ambiguity in Table Names:** When table names were similar (e.g., `employee` vs. `employees`), the model sometimes included both or selected the less relevant one, especially if the definitions were not sufficiently distinct.
- **Incomplete or Overly Broad Answers:** For queries that were vague or open-ended, the model occasionally returned all tables or omitted necessary ones, indicating a reliance on the clarity of both table definitions and user questions.
- **Hallucination:** In rare cases, the model referenced tables not present in the definitions, especially if the prompt structure was inconsistent or if the question implied a table that did not exist.

#### Lessons Learned

- **Prompt Engineering is Critical:** The specificity and clarity of both table definitions and user queries significantly impact the model's accuracy. Well-structured prompts with clear instructions yield better results.
- **Model Limitations:** While GPT-3.5-Turbo can reason about table selection, it is not infallible. It may hallucinate or misinterpret ambiguous input, especially with overlapping or unclear table names.
- **Importance of Definitions:** Detailed and distinct table definitions help the model differentiate between similar tables and reduce errors.
- **JSON Output Reliability:** The model generally followed the instruction to return results in JSON format, but occasional formatting issues occurred, especially with more complex prompts.

### Conclusion

GPT-3.5-Turbo shows promise in automating the selection of SQL tables based on natural language queries and table definitions. However, careful prompt design and clear table descriptions are essential to minimize errors and hallucinations. This approach can streamline query generation but should be supplemented with validation steps to ensure accuracy in production settings.